# Lesson 2.1 — Forward Velocity Kinematics
**Module 6 · Unit 2 · Lesson 5**

The tool twist is the rate-weighted **sum of per-joint pushes**: $\boldsymbol\xi = J(\mathbf q)\dot{\mathbf q} = \dot q_1 J_1 + \cdots + \dot q_n J_n$. We verify this on the planar 2R arm, confirm pose-dependence, and check against finite differences. Convention D-057: $[v;\omega]$, base frame.

In [1]:
import numpy as np

def skew(v):
    v=np.asarray(v,float).ravel()
    return np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
def vee(S): return np.array([S[2,1],S[0,2],S[1,0]])

def dh(theta,d,a,alpha):
    # one Denavit-Hartenberg link transform
    ct,st,ca,sa=np.cos(theta),np.sin(theta),np.cos(alpha),np.sin(alpha)
    return np.array([[ct,-st*ca, st*sa, a*ct],
                     [st, ct*ca,-ct*sa, a*st],
                     [0,  sa,    ca,    d   ],
                     [0,  0,     0,     1   ]])

def forward_chain(params,jtypes,q):
    # returns T_0^0..T_0^n ; revolute adds q to theta, prismatic adds q to d
    T=np.eye(4); Ts=[T.copy()]
    for i,(th0,d0,a,al) in enumerate(params):
        th,d=(th0+q[i],d0) if jtypes[i]=="R" else (th0,d0+q[i])
        T=T@dh(th,d,a,al); Ts.append(T.copy())
    return Ts

def geometric_jacobian(params,jtypes,q):
    # base-frame 6xn geometric Jacobian (D-057: [v; omega], linear on top)
    Ts=forward_chain(params,jtypes,q); o_n=Ts[-1][:3,3]; n=len(q); J=np.zeros((6,n))
    for i in range(n):
        z=Ts[i][:3,2]; o=Ts[i][:3,3]   # axis & origin of frame i (= z_{i-1}, o_{i-1})
        if jtypes[i]=="R": J[:3,i]=np.cross(z,o_n-o); J[3:,i]=z   # revolute: [z x (o_n-o); z]
        else:             J[:3,i]=z;                   J[3:,i]=0   # prismatic: [z; 0]
    return J

def fk_pose(params,jtypes,q):
    T=forward_chain(params,jtypes,q)[-1]; return T[:3,3], T[:3,:3]


## Planar 2R arm: build $J_v$ two ways and confirm they agree

In [2]:
checks=[]
L1=L2=1.0
params=[(0,0,L1,0),(0,0,L2,0)]; jtypes=["R","R"]

def Jv_symbolic(th):
    t1,t2=th; s1,s12=np.sin(t1),np.sin(t1+t2); c1,c12=np.cos(t1),np.cos(t1+t2)
    return np.array([[-L1*s1-L2*s12,-L2*s12],[L1*c1+L2*c12,L2*c12]])

th=np.array([0.0,np.pi/2])
Jv_geo=geometric_jacobian(params,jtypes,th)[:2,:]   # x,y rows of the geometric J
print("J_v at (0, pi/2):\n",Jv_geo)
checks.append(np.allclose(Jv_geo,Jv_symbolic(th)))
print("geometric == symbolic:",checks[-1])

J_v at (0, pi/2):
 [[-1. -1.]
 [ 1.  0.]]
geometric == symbolic: True


## The sum-of-pushes picture: both joints = column1*q̇1 + column2*q̇2

In [3]:
th=np.array([0.3,0.7]); Jv=geometric_jacobian(params,jtypes,th)[:2,:]
dq=np.array([0.5,-1.2])                       # joint rates
tool_both = Jv@dq
tool_sum  = dq[0]*Jv[:,0] + dq[1]*Jv[:,1]     # add the two pushes
print("both joints :",tool_both)
print("sum of pushes:",tool_sum)
checks.append(np.allclose(tool_both,tool_sum))

both joints : [0.44126959 0.09945663]
sum of pushes: [0.44126959 0.09945663]


## Pose-dependence: the pushes (columns) change as the arm moves

In [4]:
Ja=geometric_jacobian(params,jtypes,np.array([0.0,0.5]))[:2,:]
Jb=geometric_jacobian(params,jtypes,np.array([1.2,0.5]))[:2,:]
print("different poses give different columns:",not np.allclose(Ja,Jb))
checks.append(not np.allclose(Ja,Jb))

different poses give different columns: True


## Cross-check against finite differences of forward kinematics

In [5]:
def fk_xy(th): 
    p,_=fk_pose(params,jtypes,th); return p[:2]
eps=1e-6; th=np.array([0.3,0.7]); n=2; Jfd=np.zeros((2,n))
for i in range(n):
    e=np.zeros(n); e[i]=eps
    Jfd[:,i]=(fk_xy(th+e)-fk_xy(th-e))/(2*eps)
checks.append(np.allclose(Jfd,geometric_jacobian(params,jtypes,th)[:2,:],atol=1e-6))
print("finite-difference matches:",checks[-1])

finite-difference matches: True


In [6]:
assert all(checks), f"FAILED: {checks}"
print("All checks passed.")

All checks passed.
